In [ ]:
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
)

In [ ]:
# =========================
# 1. 配置
# =========================
class CFG:
    model_name = "microsoft/mdeberta-v3-base"
    max_len = 512
    batch_size = 8
    num_labels = 2
    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


LABEL2ID = {
    "model_a": 0,
    "model_b": 1,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
# =========================
# 2. 固定随机种子
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [ ]:
# =========================
# 3. 构造输入文本
# =========================
def build_input_text(prompt, response_a, response_b):
    text = (
        f"Prompt: {prompt}\n\n"
        f"Response A: {response_a}\n\n"
        f"Response B: {response_b}"
    )
    return text


In [ ]:
# =========================
# 4. 读取测试集
# =========================
def load_test_data(test_path):
    df = pd.read_parquet(test_path).copy()

    df["input_text"] = df.apply(
        lambda x: build_input_text(x["prompt"], x["response_a"], x["response_b"]),
        axis=1
    )
    return df

In [ ]:
# =========================
# 5. tokenizer
# =========================
def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
    return tokenizer


In [ ]:
# =========================
# 6. 测试集 Dataset
# =========================
class ArenaTestDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoded = self.tokenizer(
            row["input_text"],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }

        if "token_type_ids" in encoded:
            item["token_type_ids"] = encoded["token_type_ids"].squeeze(0)

        return item


In [ ]:
# =========================
# 7. 构造模型
# =========================
def get_model():
    config = AutoConfig.from_pretrained(CFG.model_name)
    config.num_labels = CFG.num_labels

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name,
        config=config
    )
    return model

In [ ]:
# =========================
# 8. 单个 fold 预测概率
# =========================
@torch.no_grad()
def predict_probs(model, loader, device):
    model.eval()
    all_probs = []

    for batch in tqdm(loader, total=len(loader), desc="Predicting"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        probs = torch.softmax(outputs.logits, dim=-1)
        all_probs.extend(probs.cpu().numpy())

    return np.array(all_probs)


In [ ]:

# =========================
# 9. 5折 ensemble 生成 submission.csv
# =========================
def make_submission_ensemble(model_paths, test_path, save_path="submission.csv"):
    set_seed(CFG.seed)

    tokenizer = get_tokenizer()
    test_df = load_test_data(test_path)

    test_dataset = ArenaTestDataset(test_df, tokenizer, CFG.max_len)
    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=2
    )

    all_fold_probs = []

    for model_path in model_paths:
        print(f"\nLoading model: {model_path}")
        model = get_model()

        state = torch.load(model_path, map_location=CFG.device)

        # 兼容两种保存方式
        if isinstance(state, dict) and "model_state_dict" in state:
            model.load_state_dict(state["model_state_dict"])
        else:
            model.load_state_dict(state)

        model = model.to(CFG.device)

        fold_probs = predict_probs(model, test_loader, CFG.device)
        all_fold_probs.append(fold_probs)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_probs = np.mean(all_fold_probs, axis=0)
    pred_ids = mean_probs.argmax(axis=1)
    pred_labels = [ID2LABEL[int(x)] for x in pred_ids]

    submission = pd.DataFrame({
        "id": test_df["id"].values,
        "winner": pred_labels
    })
    submission.to_csv(save_path, index=False)
    print(f"\nsubmission saved to: {save_path}")

    # 可选：保存带概率的版本，方便你自己检查
    detail_df = pd.DataFrame({
        "id": test_df["id"].values,
        "prob_model_a": mean_probs[:, 0],
        "prob_model_b": mean_probs[:, 1],
        "winner": pred_labels
    })
    detail_df.to_csv("submission_with_probs.csv", index=False)
    print("detailed prediction saved to: submission_with_probs.csv")

    return submission, detail_df

In [ ]:
# =========================
# 10. 主函数
# =========================
if __name__ == "__main__":
    test_path = "/kaggle/input/datasets/ai2334135131/train-data/test.parquet"

    model_paths = [
        "/kaggle/working/checkpoints/baseline_fold0.pth",
        "/kaggle/working/checkpoints/baseline_fold1.pth",
        "/kaggle/working/checkpoints/baseline_fold2.pth",
        "/kaggle/working/checkpoints/baseline_fold3.pth",
        "/kaggle/working/checkpoints/baseline_fold4.pth",
    ]

    submission, detail_df = make_submission_ensemble(
        model_paths=model_paths,
        test_path=test_path,
        save_path="submission.csv"
    )

    print(submission.head())